# 009 — Cross-Encoder Reranker
**Retrieval Series** | Two-stage: fast recall → precise reranking

**Bi-encoder** (vector search): Fast, approximate — ANN search over millions of docs.
**Cross-encoder** (reranker): Slow but precise — reads query + doc together, full attention.

Use bi-encoder to recall top-50, then cross-encoder to rerank to top-5.


In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── Build initial retriever (stage 1) ────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
docs = [Document(page_content=c)
        for text in [ai_text, ml_text, dl_text, nlp_text]
        for c in splitter.split_text(text[:15000])]

vectorstore = FAISS.from_documents(docs, embeddings)
# Retrieve more candidates than needed (will rerank to fewer)
initial_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})
print(f"✓ Stage-1 retriever over {len(docs)} docs (returns top-20)")


In [ ]:
# ── Cross-encoder reranker (stage 2) ─────────────────────────────────────
from sentence_transformers import CrossEncoder
import numpy as np

# Load a small cross-encoder (downloads ~90MB on first run)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✓ Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, docs: list, top_k: int = 5) -> list:
    pairs   = [(query, d.page_content) for d in docs]
    scores  = cross_encoder.predict(pairs)
    ranked  = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [(score, doc) for score, doc in ranked[:top_k]]


In [ ]:
# ── Before vs After reranking ─────────────────────────────────────────────
import time

query = "How does the attention mechanism work in transformer models?"

# Stage 1: bi-encoder retrieval (fast but imprecise ordering)
t0 = time.time()
stage1_docs = initial_retriever.invoke(query)
t_stage1 = time.time() - t0
print(f"Stage 1 ({t_stage1*1000:.0f}ms): retrieved {len(stage1_docs)} candidates")

# Stage 2: cross-encoder reranking (precise)
t0 = time.time()
reranked = rerank(query, stage1_docs, top_k=5)
t_stage2 = time.time() - t0
print(f"Stage 2 ({t_stage2*1000:.0f}ms): reranked to top {len(reranked)}")

print("\n── Top-3 BEFORE reranking (bi-encoder order) ──")
for i, d in enumerate(stage1_docs[:3]):
    print(f"  [{i+1}] {d.page_content[:130]}...")

print("\n── Top-3 AFTER reranking (cross-encoder order) ──")
for i, (score, d) in enumerate(reranked[:3]):
    # Show how much the rank changed
    orig_rank = next((j+1 for j, doc in enumerate(stage1_docs)
                      if doc.page_content == d.page_content), "?")
    print(f"  [{i+1}] ce_score={score:+.2f}  (was rank #{orig_rank})")
    print(f"       {d.page_content[:130]}...")


In [ ]:
# ── Full pipeline: retrieve → rerank → generate ───────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def rag_with_reranking(question: str, final_k: int = 4):
    # Stage 1
    candidates = initial_retriever.invoke(question)
    # Stage 2
    reranked_scored = rerank(question, candidates, top_k=final_k)
    top_docs = [doc for _, doc in reranked_scored]
    ctx = "\n\n".join(d.page_content for d in top_docs)
    # Generate
    chain = (
        ChatPromptTemplate.from_template(
            "Answer using only the context.\nContext:\n{ctx}\nQuestion: {q}\nAnswer:"
        )
        | llm | StrOutputParser()
    )
    return chain.invoke({"ctx": ctx[:3500], "q": question})

questions = [
    "What is the vanishing gradient problem in deep learning?",
    "How does BERT differ from GPT in its training approach?",
]
for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_with_reranking(q)}\n")


## Key Takeaways
- **Retrieve 20, rerank to 5** — the standard two-stage pattern
- Cross-encoder score range: typically −10 to +10 (relative, not probability)
- `ms-marco-MiniLM-L-6-v2` is fast (6 layers); for better quality use `L-12-v2`
- Latency trade-off: ~20ms per (query, doc) pair on CPU — rerank 20 docs ≈ 400ms
